In [ ]:
# Example 1: the CLI entrypoint is a Typer app — confirm it exists and has commands registered.
from fastapi_cloud_cli.cli import app
import typer

assert isinstance(app, typer.Typer), f"expected typer.Typer, got {type(app)}"

# Typer apps register commands via registered_commands; sub-apps via registered_groups.
command_names = [c.name or (c.callback.__name__ if c.callback else None) for c in app.registered_commands]
group_names = [g.name for g in app.registered_groups]

print(f"Registered commands: {command_names}")
print(f"Registered sub-groups: {group_names}")

# We expect at least one command or group on the root app.
assert command_names or group_names, "fastapi_cloud_cli.app has no commands/groups registered"


In [ ]:
# Example 2: invoke the CLI with --help in a real subprocess — proves the app wires up cleanly.
# (Subprocess, not typer.testing.CliRunner: inside a Jupyter kernel typer/rich render help
# through Jupyter's display hooks, so CliRunner's stdout capture comes back empty.)
import os
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, "-m", "fastapi_cloud_cli", "--help"],
    capture_output=True,
    text=True,
    timeout=120,
    env={**os.environ, "NO_COLOR": "1"},
)

print(f"exit_code = {proc.returncode}")
print("--- stdout (first 600 chars) ---")
print(proc.stdout[:600])

assert proc.returncode == 0, f"--help exited non-zero: {proc.returncode}\n{proc.stderr}"
assert proc.stdout.strip(), "--help produced no output"

In [ ]:
# Example 3: probe the package's public submodules — these are the documented internals.
import importlib

candidate_modules = [
    "fastapi_cloud_cli.cli",
    "fastapi_cloud_cli.commands",
    "fastapi_cloud_cli.utils",
    "fastapi_cloud_cli.config",
]

imported = {name: importlib.import_module(name) for name in candidate_modules}

for name, mod in imported.items():
    print(f"{name}: OK")
    assert mod.__name__ == name, (name, mod.__name__)

In [ ]:
# Example 4: confirm the runtime deps the CLI relies on are importable + composable.
# fastapi_cloud_cli builds on typer + httpx + rich_toolkit + pydantic.
import httpx
import pydantic
import rich_toolkit
import typer

print(f"typer        = {typer.__version__}")
print(f"httpx        = {httpx.__version__}")
print(f"pydantic     = {pydantic.VERSION}")
print(f"rich_toolkit = {getattr(rich_toolkit, '__version__', 'unknown')}")

# Sanity: build a trivial pydantic model the way fastapi_cloud_cli's internal config does.
class _MiniConfig(pydantic.BaseModel):
    api_url: str = "https://api.example.invalid"
    token: str = ""

cfg = _MiniConfig(token="abc")
assert cfg.token == "abc" and cfg.api_url.startswith("https://")
print(f"\nMiniConfig roundtrip: {cfg.model_dump()}")


In [ ]:
# Example 5: distribution metadata + entrypoint chain. fastapi-cloud-cli ships as a LIBRARY
# consumed by the `fastapi` CLI (`fastapi deploy`): its own wheel declares an EMPTY
# [console_scripts] section (verified against the installed dist-info), so the reachable
# entrypoints are `python -m fastapi_cloud_cli` (Example 2) and fastapi_cloud_cli.cli:main.
import importlib.metadata as md

dist = md.distribution("fastapi-cloud-cli")
print(f"Distribution: {dist.metadata['Name']} {dist.version}")
assert dist.version, "fastapi-cloud-cli distribution has no version"

own_scripts = [
    ep for ep in md.entry_points(group="console_scripts")
    if (ep.value or "").startswith("fastapi_cloud_cli")
]
print(f"console_scripts declared by fastapi_cloud_cli: {own_scripts}")
assert not own_scripts, f"fastapi-cloud-cli 0.22.x declares no console_scripts, found: {own_scripts}"

# The callable the `fastapi` CLI dispatches to must resolve.
from fastapi_cloud_cli.cli import main

assert callable(main), "fastapi_cloud_cli.cli.main must be callable"
print("fastapi_cloud_cli.cli.main resolves — CLI reachable via `python -m fastapi_cloud_cli`")